# 00 · Hello Claude — API smoke test

One round-trip to the Claude API. The point is to prove the plumbing — key, SDK, network — before
building anything on top. If this works, every later failure inside LangGraph / ragas is *your* code,
not the connection.

**Before running:** add `ANTHROPIC_API_KEY=sk-ant-...` to `.env` in the repo root (same file as the
Companies House keys; console.anthropic.com → API Keys). `.env` is gitignored.

**Kernel:** *Python 3.13 (Lloyds .venv)*. The first cell checks that.

In [2]:
import os, sys
from pathlib import Path

import anthropic
from dotenv import load_dotenv

# A notebook has no __file__, so walk up from the cwd until we find the repo's .env.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists())
load_dotenv(ROOT / ".env")

print("interpreter:", sys.executable)
print("anthropic  :", anthropic.__version__)
print(".env       :", ROOT / ".env")
assert ".venv" in sys.executable, "wrong kernel — select 'Python 3.13 (Lloyds .venv)'"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not in .env — add it and restart the kernel"

interpreter: /Users/natchalin_/Projects/final_project/Lloyds/.venv/bin/python
anthropic  : 1.5.0
.env       : /Users/natchalin_/Projects/final_project/Lloyds/.env


## The call

`anthropic.Anthropic()` with no arguments reads `ANTHROPIC_API_KEY` from the environment — never pass the
key as a string in code. `claude-opus-5` is the current default model; it thinks adaptively by default, so
a hello-world needs no extra settings.

In [4]:
MODEL = "claude-sonnet-5"
client = anthropic.Anthropic()

try:
    response = client.messages.create(
        model=MODEL,
        max_tokens=256,
        messages=[{"role": "user", "content": "Say hello in one sentence, and name the model you are."}],
    )
except anthropic.AuthenticationError:
    raise SystemExit("Authentication failed: ANTHROPIC_API_KEY is invalid — regenerate it in the console")
except anthropic.RateLimitError as e:
    raise SystemExit(f"Rate limited; retry after {e.response.headers.get('retry-after', '?')}s")
except anthropic.BadRequestError as e:
    raise SystemExit(f"API rejected the request: {e.message}")   # e.g. no credit on the account
except anthropic.APIConnectionError:
    raise SystemExit("Could not reach api.anthropic.com — check the network")

text = next(b.text for b in response.content if b.type == "text")
print(f"model      : {response.model}")
print(f"stop_reason: {response.stop_reason}")
print(f"reply      : {text}")
u = response.usage
print(f"usage      : {u.input_tokens} in / {u.output_tokens} out tokens")

model      : claude-sonnet-5
stop_reason: end_turn
reply      : Hello! I'm Claude, an AI assistant made by Anthropic.
usage      : 23 in / 27 out tokens


## What "working" looks like

```
model      : claude-opus-5
stop_reason: end_turn
reply      : Hello! I'm Claude, ...
usage      : ~20 in / ~30 out tokens
```

Get in the habit of reading `usage` now — when ragas is scoring hundreds of leads, that number is the bill.

Two things deliberately left out of a smoke test that belong in the real agent: Opus 5 can return
`stop_reason: "refusal"` (the API has a `fallbacks` parameter to reroute those), and long outputs should
use `client.messages.stream(...)` rather than `create`. Neither matters for one sentence.